# Fine-tune PP-OCRv6 recognition on handwritten C code (Colab)

Clean, re-runnable version of the working sequence. Trains `PP-OCRv6_medium_rec`
on the line-crop dataset built by `evaluators.build_recognition_dataset`.

**Result this produced (2026-08-30):** held-out CER `clean_ws` **0.274 -> 0.126**
(stock vs fine-tuned, same 20-image test set). See `docs/ocr/EVALUATION.md`.

**Note on the repo path:** the original session got the PaddleOCR repo via
`paddlex --install`, whose dependency install is broken on Colab's Python 3.13.
This clean notebook uses `git clone` instead (same repo, `/content/PaddleOCR`) and
runs `tools/train.py` directly. Full rationale: `docs/ocr/COLAB_SETUP_WORKING.md`.

**Before running:** Runtime -> Change runtime type -> **GPU** (T4 is enough).

## Part A - Environment

In [ ]:
!nvidia-smi

In [ ]:
# PaddlePaddle GPU build is not on plain PyPI - use Paddle's own index (cu130 = CUDA 13.0).
!python -m pip install -q paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu130/

In [ ]:
import paddle
print(paddle.__version__)
paddle.utils.run_check()

In [ ]:
# Clean way to get the repo that contains tools/train.py (bypasses the broken paddlex installer).
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR /content/PaddleOCR

In [ ]:
# The two extra runtime deps tools/train.py needs beyond Colab's base image.
# If a later run reports ModuleNotFoundError: X, just `!pip install -q X` and re-run.
!pip install -q lmdb rapidfuzz

## Part B - Upload the dataset

Upload `recognition_dataset.zip` (built locally by `evaluators.build_recognition_dataset`).

In [ ]:
from google.colab import files
uploaded = files.upload()   # select recognition_dataset.zip

In [ ]:
!unzip -q recognition_dataset.zip -d /content/
!ls /content/datasets/recognition/
!wc -l /content/datasets/recognition/train.txt /content/datasets/recognition/val.txt

## Part C - Pretrained weights + config fix (Cell A)

Download the pretrained weights locally (native PaddleOCR won't fetch a URL for
`Global.pretrained_model`) and raise `max_text_length` 25 -> 100 (at 25, ~30% of
our lines, up to 92 chars, would be silently dropped).

In [ ]:
import os, subprocess

REPO = "/content/PaddleOCR"
CFG = f"{REPO}/configs/rec/PP-OCRv6/PP-OCRv6_medium_rec.yml"

# 1) pretrained weights (~232 MB; a tiny size here means the URL failed)
url = "https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_medium_rec_pretrained.pdparams"
dst = "/content/PP-OCRv6_medium_rec_pretrained.pdparams"
subprocess.run(["wget", "-q", "-O", dst, url], check=True)
print("pretrained bytes:", os.path.getsize(dst))

# 2) max_text_length 25 -> 100 at the YAML anchor (propagates everywhere)
with open(CFG) as f:
    txt = f.read()
assert "&max_text_length 25" in txt, "anchor not found - config changed?"
with open(CFG, "w") as f:
    f.write(txt.replace("&max_text_length 25", "&max_text_length 100"))
print("max_text_length patched to 100")

## Part D - Train (Cell B)

Overrides: `epoch_num=40`, single-GPU (`distributed=false`), transfer-learn from
the pretrained weights, `ratio_list=[1.0]` (config default 0.5 would use half the
crops), `num_workers=2` (Colab has ~2 CPUs). Training has started when the
tracebacks stop and `ppocr INFO: epoch: ... loss:` lines appear.

In [ ]:
%cd /content/PaddleOCR
!python tools/train.py -c configs/rec/PP-OCRv6/PP-OCRv6_medium_rec.yml \
  -o Global.epoch_num=40 Global.distributed=false \
     Global.pretrained_model=/content/PP-OCRv6_medium_rec_pretrained \
     Global.save_model_dir=/content/output/PP-OCRv6_medium_rec \
     Global.save_epoch_step=5 Global.eval_batch_step=[0,200] \
     Train.dataset.data_dir=/content/datasets/recognition \
     Train.dataset.label_file_list=[/content/datasets/recognition/train.txt] \
     Train.dataset.ratio_list=[1.0] \
     Train.loader.num_workers=2 \
     Eval.dataset.data_dir=/content/datasets/recognition \
     Eval.dataset.label_file_list=[/content/datasets/recognition/val.txt] \
     Eval.loader.num_workers=2

## Part E - Export + download

Export the best checkpoint to inference format, then download it. Unzip locally
into `ocr_feature/models/fine_tuned_rec/inference/` and point
`core/ocr_pipeline.py` at it (see `docs/ocr/COLAB_SETUP_WORKING.md` Part E).

In [ ]:
%cd /content/PaddleOCR
!python tools/export_model.py -c configs/rec/PP-OCRv6/PP-OCRv6_medium_rec.yml \
  -o Global.pretrained_model=/content/output/PP-OCRv6_medium_rec/best_accuracy \
     Global.save_inference_dir=/content/inference/PP-OCRv6_medium_rec

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/fine_tuned_rec_model", "zip", "/content/inference/PP-OCRv6_medium_rec")
files.download("/content/fine_tuned_rec_model.zip")